In [ ]:
# =========================
# signlessCorr_MDS: 1 - |cor| 距離 × 線形(cmdscale)/非線形(isoMDS)
# - OH/FP を一括処理
# - top3 / cumeig(>=0.8) の両条件
# - NbClust (ward.D2, Euclidean) × 6指標
# - 出力: signlessCorr_MDS_YYYYMMDD/<OH|FP>/ に
#   寄与CSV / 座標CSV(top3,cumeig, linear/nonlinear) / 割当CSV / 図 / ARI / ラベル一括CSV・kサマリ を保存
# =========================

if (!require(NbClust)) { install.packages("NbClust"); library(NbClust) }
if (!require(ggplot2)) { install.packages("ggplot2"); library(ggplot2) }
if (!require(ggrepel)) { install.packages("ggrepel"); library(ggrepel) }
if (!require(scales))  { install.packages("scales");  library(scales) }
if (!require(MASS))    { install.packages("MASS");    library(MASS) }     # isoMDS
if (!require(mclust))  { install.packages("mclust");  library(mclust) }   # ARI
if (!require(dplyr))   { install.packages("dplyr");   library(dplyr) }

# ---- 設定 ----
base_dir  <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No"
dataset_files <- c("preprocessed_features_OH.csv", "preprocessed_features_FP.csv")
index_list    <- c("silhouette","dunn","gap","ch","db","ptbiserial")
modes         <- c("linear","nonlinear")
dim_modes     <- c("top3","cumeig")
target_vars   <- c("PCEmax","Jsc","Voc","FF")

set.seed(42)
ts_day   <- format(Sys.Date(), "%Y%m%d")
root_out <- file.path(base_dir, paste0("STEP3.2_signlessCorr_MDS_", ts_day))  # ★違いが分かる命名
dir.create(root_out, showWarnings = FALSE, recursive = TRUE)

pct <- function(x) sprintf("%.2f%%", 100*x)

read_numeric_explanatory <- function(path, target_vars){
  df <- read.delim(path, header = TRUE, sep = ",", row.names = 1,
                   as.is = TRUE, strip.white = FALSE)
  # 目的変数を確実に除外
  expl_vars <- df[, !(colnames(df) %in% target_vars), drop = FALSE]
  # 文字列列の粗除去（英字を含むものを除外）
  is_char <- sapply(expl_vars, function(col) any(grepl("[A-Za-z]", col) & col != "NA", na.rm = TRUE))
  numData <- expl_vars[, !is_char, drop = FALSE]
  if (!ncol(numData)) stop("数値の説明変数列が見つかりません: ", path)
  return(numData)
}

# signless 相関距離と MDS(線形) の準備
prep_signless_linear_mds <- function(numData, k_max = 200){
  corM <- suppressWarnings(cor(numData, use = "pairwise.complete.obs"))
  corM[is.na(corM)] <- 0
  dist_mat <- 1 - abs(corM)
  dist_mat[dist_mat <= 0] <- 1e-6  # 0 距離の微小補正
  ddata <- as.dist(dist_mat)

  n_vars <- ncol(numData)
  cmd_k  <- min(max(1, n_vars - 1), k_max)
  mds_linear <- cmdscale(ddata, eig = TRUE, k = cmd_k)

  # 正の固有値で寄与率
  all_eigs <- mds_linear$eig
  pos_idx  <- which(all_eigs > 0)
  pos_eigs <- all_eigs[pos_idx]
  contrib  <- if (length(pos_eigs) > 0) pos_eigs / sum(pos_eigs) else numeric(0)
  list(ddata = ddata, mds_linear = mds_linear,
       all_eigs = all_eigs, contrib_pos = contrib)
}

# cumeig(>=0.8) の軸数（最低3軸）を決める
decide_k_cumeig <- function(contrib_pos, thr = 0.8){
  if (length(contrib_pos) == 0) return(3)
  k <- which(cumsum(contrib_pos) >= thr)[1]
  if (is.na(k) || k < 2) k <- 3
  return(k)
}

# NbClust を安全実行
run_nbclust <- function(X, index){
  tryCatch({
    NbClust(data = X, diss = NULL, distance = "euclidean",
            min.nc = 2, max.nc = min(100, nrow(X) - 1),
            method = "ward.D2", index = index)
  }, error = function(e) { warning(paste("NbClust failed:", index, "-", e$message)); NULL })
}

# 図保存（スクリープロット / Top50）
save_eig_plots <- function(out_dir, suffix_tag, ts_tag, contrib_pos){
  if (length(contrib_pos) == 0) { warning("No positive eigenvalues; plots skipped."); return() }
  df <- data.frame(Dim = seq_along(contrib_pos),
                   Prop = as.numeric(contrib_pos))
  df$Cum <- cumsum(df$Prop)

  p1 <- ggplot(df, aes(Dim, Prop)) +
    geom_col(width = 0.85, alpha = 0.9) +
    geom_line(aes(y = Cum), linewidth = 1.1) +
    geom_point(aes(y = Cum), size = 2) +
    scale_y_continuous(labels = scales::percent_format(accuracy = 1),
                       sec.axis = sec_axis(~ ., name = "Cumulative share")) +
    labs(title = "Scree plot (positive eigenvalues; signlessCorr distance)",
         subtitle = suffix_tag, x = "Dimension", y = "Variance share") +
    theme_minimal(base_size = 12) +
    theme(panel.grid.minor = element_blank(),
          axis.title = element_text(face = "bold"),
          plot.title = element_text(face = "bold"))
  ggsave(file.path(out_dir, paste0("MDS_scree_", suffix_tag, "_", ts_tag, ".png")),
         p1, width = 7, height = 5, dpi = 300)

  topN <- min(50, nrow(df))
  p2 <- ggplot(df[1:topN, ], aes(Dim, Prop)) +
    geom_col(width = 0.85, alpha = 0.95) +
    labs(title = paste0("Top-", topN, " eigenvalue shares (signlessCorr)"),
         subtitle = suffix_tag, x = "Dimension", y = "Variance share") +
    scale_y_continuous(labels = scales::percent_format(accuracy = 1)) +
    theme_minimal(base_size = 12) +
    theme(panel.grid.minor = element_blank(),
          axis.title = element_text(face = "bold"),
          plot.title = element_text(face = "bold"))
  ggsave(file.path(out_dir, paste0("MDS_bar_top", topN, "_", suffix_tag, "_", ts_tag, ".png")),
         p2, width = 7, height = 5, dpi = 300)
}

# ========= メイン：OH / FP を順に実行 =========
for (ifname in dataset_files) {
  suffix_tag <- if (grepl("OH", ifname, ignore.case = TRUE)) "OH" else "FP"
  out_dir <- file.path(root_out, suffix_tag)
  dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)
  ts_tag <- format(Sys.time(), "%Y%m%d_%H%M%S")

  message("\n==============================")
  message(sprintf("=== Processing: %s (%s) ===", ifname, suffix_tag))
  message("==============================")

  # 1) データ読み込み（説明変数のみ、英字列除外）
  in_path <- file.path(base_dir, ifname)
  numData <- read_numeric_explanatory(in_path, target_vars)

  # 2) 距離=1-|cor| → 線形MDS（eig付き） → 正固有値寄与
  prep <- prep_signless_linear_mds(numData)
  ddata       <- prep$ddata
  mds_linear  <- prep$mds_linear
  all_eigs    <- prep$all_eigs
  contrib_pos <- prep$contrib_pos

  # 3) cumeig(>=0.8) 軸数
  k_cumeig <- decide_k_cumeig(contrib_pos, thr = 0.8)

  # 4) 固有値寄与 CSV
  df_eig_out <- data.frame(
    Dim           = seq_along(all_eigs),
    Eigenvalue    = all_eigs,
    Share_pos     = { z <- rep(NA_real_, length(all_eigs)); 
                      pos <- which(all_eigs > 0); 
                      if (length(pos)) z[pos] <- all_eigs[pos] / sum(all_eigs[pos]); z },
    CumShare_pos  = { z <- rep(NA_real_, length(all_eigs));
                      pos <- which(all_eigs > 0);
                      if (length(pos)) { tmp <- all_eigs[pos]/sum(all_eigs[pos]); z[pos] <- cumsum(tmp) } ; z }
  )
  write.csv(df_eig_out, file.path(out_dir, paste0("MDS_eigen_contributions_", suffix_tag, "_", ts_tag, ".csv")),
            row.names = FALSE)

  # 5) スクリープロット等
  save_eig_plots(out_dir, suffix_tag, ts_tag, contrib_pos)

  # 6) 座標計算（linear/nonlinear × top3/cumeig）
  coords_list <- list()
  # linear: 条件ごとに cmdscale を再計算（提示コードの流儀）
  coords_list[["linear_top3"]]   <- cmdscale(ddata, eig = FALSE, k = 3)
  coords_list[["linear_cumeig"]] <- cmdscale(ddata, eig = FALSE, k = k_cumeig)
  # nonlinear: isoMDS を top3 / cumeig の両方で（比較できるよう拡張）
  coords_list[["nonlinear_top3"]]   <- MASS::isoMDS(ddata, k = 3)$points
  coords_list[["nonlinear_cumeig"]] <- MASS::isoMDS(ddata, k = k_cumeig)$points

  # 行名は変数名に
  for (nm in names(coords_list)) rownames(coords_list[[nm]]) <- colnames(numData)

  # 座標CSVを保存
  for (nm in names(coords_list)) {
    write.csv(coords_list[[nm]],
              file.path(out_dir, paste0("MDScoords_", nm, "_", suffix_tag, "_", ts_tag, ".csv")))
  }

  # 7) クラスタ割当のまとめテーブルと k サマリ
  var_ids <- colnames(numData)
  label_df <- data.frame(id = var_ids, stringsAsFactors = FALSE)
  ksum <- data.frame(mode = character(0), dim_mode = character(0),
                     index = character(0), k = integer(0), stringsAsFactors = FALSE)

  # 8) 各条件×指標で NbClust 実行・図保存・割当保存
  for (cond in names(coords_list)) {
    X <- coords_list[[cond]]
    mm <- strsplit(cond, "_")[[1]]; mode <- mm[1]; dim_mode <- mm[2]

    for (idx in index_list) {
      res <- run_nbclust(X, idx)
      col_name <- paste(cond, idx, sep = "_")
      if (is.null(res)) {
        label_df[[col_name]] <- NA_integer_
        ksum <- rbind(ksum, data.frame(mode = mode, dim_mode = dim_mode, index = idx, k = NA_integer_))
        next
      }

      # 割当
      grp <- as.factor(res$Best.partition)
      grp <- grp[var_ids]  # 行順合わせ
      label_df[[col_name]] <- as.integer(grp)

      # k
      k_val <- NA_integer_
      if (!is.null(res$Best.nc)) {
        if (is.list(res$Best.nc) && !is.null(res$Best.nc[["Number_clusters"]])) {
          k_val <- as.integer(res$Best.nc[["Number_clusters"]])
        } else if (is.matrix(res$Best.nc) && "Number_clusters" %in% rownames(res$Best.nc)) {
          k_val <- as.integer(res$Best.nc["Number_clusters", 1])
        } else if (is.numeric(res$Best.nc)) {
          k_val <- as.integer(res$Best.nc[1])
        }
      }
      ksum <- rbind(ksum, data.frame(mode = mode, dim_mode = dim_mode, index = idx, k = k_val))

      # 割当CSV（個別）
      write.csv(data.frame(Variable = names(grp), Cluster = grp),
                file.path(out_dir, paste0("ClusterAssign_", cond, "_", idx, "_", suffix_tag, "_", ts_tag, ".csv")),
                row.names = FALSE)

      # All.index プロット
      if (!is.null(res$All.index)) {
        idx_vals <- as.numeric(res$All.index)
        k_labels <- names(res$All.index)
        k_seq <- if (is.null(k_labels) || any(k_labels == "")) seq_along(idx_vals) else suppressWarnings(as.numeric(k_labels))
        if (any(is.na(k_seq))) k_seq <- seq_along(idx_vals)
        df_idx <- data.frame(k = k_seq, value = idx_vals)
        p_idx <- ggplot(df_idx, aes(k, value)) +
          geom_line(linewidth = 1) + geom_point(size = 2) +
          labs(title = paste0("NbClust All.index — ", idx),
               subtitle = paste0(mode, " / ", dim_mode, "  |  ", suffix_tag),
               x = "k", y = "Index value") +
          theme_minimal(base_size = 12) +
          theme(panel.grid.minor = element_blank(),
                axis.title = element_text(face = "bold"),
                plot.title = element_text(face = "bold"))
        ggsave(file.path(out_dir, paste0("NbClust_AllIndex_", cond, "_", idx, "_", suffix_tag, "_", ts_tag, ".png")),
               p_idx, width = 7, height = 5, dpi = 300)
      }

      # 1-2軸散布図
      df_plot <- data.frame(MDS1 = X[,1], MDS2 = X[,2], Cluster = grp, ID = rownames(X))
      rng <- range(c(df_plot$MDS1, df_plot$MDS2), na.rm = TRUE)
      pad <- diff(rng) * 0.05; lims <- c(min(rng) - pad, max(rng) + pad)
      p_sc <- ggplot(df_plot, aes(MDS1, MDS2, color = Cluster)) +
        stat_ellipse(aes(group = Cluster), type = "norm", level = 0.95, linewidth = 0.6, alpha = 0.25) +
        geom_point(size = 2.2, alpha = 0.8) +
        coord_equal(xlim = lims, ylim = lims, expand = TRUE) +
        scale_color_hue(h = c(0,360), l = 55, c = 90) +
        labs(title = paste0("MDS (", mode, "/", dim_mode, ") — Ward.D2 × NbClust"),
             subtitle = paste0("Index: ", idx, "  |  k = ", k_val, "  |  ", suffix_tag),
             x = "Dim 1", y = "Dim 2", color = "Cluster") +
        theme_minimal(base_size = 12) +
        theme(panel.grid.major = element_line(linewidth = 0.3, linetype = "dashed"),
              panel.grid.minor = element_blank(),
              axis.title = element_text(face = "bold"),
              plot.title = element_text(face = "bold"))
      ggsave(file.path(out_dir, paste0("MDS12_scatter_", cond, "_", idx, "_", suffix_tag, "_", ts_tag, ".png")),
             p_sc, width = 7, height = 6, dpi = 300)
    }
  }

  # 9) ラベル一括CSV・kサマリCSV（new と整合）
  write.csv(label_df, file.path(out_dir, paste0("cluster_labels_", suffix_tag, "_", ts_tag, ".csv")), row.names = FALSE)
  write.csv(ksum,     file.path(out_dir, paste0("cluster_k_summary_", suffix_tag, "_", ts_tag, ".csv")), row.names = FALSE)

  # 10) ARI: top3 vs cumeig を mode 別に評価（linear / nonlinear）
  for (mode in c("linear","nonlinear")) {
    ari_rows <- lapply(index_list, function(ix){
      a <- label_df[[paste0(mode, "_top3_", ix)]]
      b <- label_df[[paste0(mode, "_cumeig_", ix)]]
      if (is.null(a) || is.null(b) || all(is.na(a)) || all(is.na(b))) {
        data.frame(Index = ix, ARI_top3_vs_cumeig = NA_real_)
      } else {
        data.frame(Index = ix,
                   ARI_top3_vs_cumeig = mclust::adjustedRandIndex(as.factor(a), as.factor(b)))
      }
    })
    df_ari <- dplyr::bind_rows(ari_rows)
    write.csv(df_ari, file.path(out_dir, paste0("ARI_top3_vs_cumeig_", mode, "_", suffix_tag, "_", ts_tag, ".csv")),
              row.names = FALSE)
  }

  cat("\n✅ Done:", suffix_tag, "→", normalizePath(out_dir), "\n")
}

cat("\n🎉 All finished → ", normalizePath(root_out), "\n")


Loading required package: NbClust

Loading required package: ggplot2

Loading required package: ggrepel

Loading required package: scales

Loading required package: MASS

Loading required package: mclust

Package 'mclust' version 6.1.1
Type 'citation("mclust")' for citing this R package in publications.

Loading required package: dplyr


Attaching package: 'dplyr'


The following object is masked from 'package:MASS':

    select


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union




=== Processing: preprocessed_features_OH.csv (OH) ===


